## Workflow for Neosurf-on-Neosurf MaSIF search

In [54]:
import os
import pandas as pd
import sys
import numpy as np
import subprocess

repo_root = !git rev-parse --show-toplevel
repo_root = repo_root[0]
os.chdir(repo_root)


# ----- Source code and scripts ------
# Add source_dir to python PATH
source_dir = os.path.join(repo_root, 'masif_seed_search/source')
sys.path.insert(0, source_dir)

# Add python_scripts_dir to python PATH
python_scripts_dir = os.path.join(repo_root, 'scripts/python')
sys.path.insert(0, python_scripts_dir)

prepare_input_py = os.path.join(repo_root, 'scripts/python/prepare_input.py')

# Settings:
N_ARRAY_JOBS = 500
EVOEF2_BIN = os.path.join(repo_root, "EvoEF2/EvoEF2")
N_SEED = None


# ----- Directories ------
data_dir = os.path.join(repo_root, 'data')

# ----- Input ------
# .csv about the seed complexes - use all human reference proteome liganded pdbs
seed_list_csv = os.path.join(data_dir, "human_reference_proteome_liganded_pdbs/human_reference_proteome_pdb_ligands_split.csv")

# Directory to store input .pdb and .sdf
input_dir = os.path.join(data_dir, 'input')
os.makedirs(input_dir, exist_ok=True)

# ----- Processing directory ------
processing_dir = os.path.join(data_dir, 'processing')
os.makedirs(processing_dir, exist_ok=True)

# 1. Prepare input files
prep_input_proc_dir = os.path.join(processing_dir, '1_prep_input')
os.makedirs(prep_input_proc_dir, exist_ok=True)

# 2. Run MaSIF preprocessing
masif_preprocess_proc_dir = os.path.join(processing_dir, '2_masif_preprocess')
os.makedirs(masif_preprocess_proc_dir, exist_ok=True)

# 3. Run MaSIF search
masif_search_proc_dir = os.path.join(processing_dir, '3_masif_search')
os.makedirs(masif_search_proc_dir, exist_ok=True)

# ----- Output ------
# Directory to write preprocessing files
preprocess_dir = os.path.join(data_dir, 'preprocess')
os.makedirs(preprocess_dir, exist_ok=True)

# Directory to write masif-search output
masif_search_out_dir = os.path.join(data_dir, 'masif_search')
os.makedirs(masif_search_out_dir, exist_ok=True)

master_subset_dir = os.path.join(masif_search_out_dir, 'subset')
os.makedirs(master_subset_dir, exist_ok=True)

query_targets_list = os.path.join(masif_search_out_dir, 'query_targets.txt')



___
### Step 1 - Prepare input .pdb and .sdf files

In [47]:
df_seed = pd.read_csv(seed_list_csv)

# Use only the first N_SEED rows for testing 
if N_SEED is not None:
    df_seed = df_seed.head(N_SEED)
    
print(f"df_seed.shape: {df_seed.shape}")
df_seed.head()

df_seed.shape: (15233, 20)


,uniprot_id,gene_name,recommendedName,pdb_id,protein_chain,ligand_chain,ligand_code,ligand_name,smiles,formula,mw,qed,num_carbon,num_N_O,uniprot_id_count,method,resolution,neighbor_count,percent_intracellular,split
0,Q8TBX8,PIP4K2C,Phosphatidylinositol 5-phosphate 4-kinase type...,7QPN,B,B,DVF,5-methyl-2-(2-propan-2-ylphenyl)-~{N}-(pyridin...,CC(C)c1ccccc1c2nc3ccn(c3c(n2)NCc4ccccn4)C,C22H23N5,357.195346,0.559088,22.0,5.0,1.0,X-RAY DIFFRACTION,1.95,32.0,1.0,seed
1,Q15562,TEAD2,Transcriptional enhancer factor TEF-4,8CUH,B,B,P0I,4-[3-(2-cyclohexylethoxy)benzoyl]-N-phenylpipe...,c1ccc(cc1)NC(=O)N2CCN(CC2)C(=O)c3cccc(c3)OCCC4...,C26H33N3O3,435.252192,0.692559,26.0,6.0,1.0,X-RAY DIFFRACTION,2.40,28.0,1.0,seed
2,Q07869,PPARA,Peroxisome proliferator-activated receptor alpha,2P54,A,A,735,2-METHYL-2-(4-{[({4-METHYL-2-[4-(TRIFLUOROMETH...,Cc1c(sc(n1)c2ccc(cc2)C(F)(F)F)C(=O)NCc3ccc(cc3...,C23H21F3N2O4S,478.117413,0.480598,23.0,6.0,1.0,X-RAY DIFFRACTION,1.79,28.0,1.0,seed
3,P62508,ESRRG,Estrogen-related receptor gamma,6A6K,B,B,9S6,3-[(~{E})-5-oxidanyl-2-phenyl-1-[4-(4-propan-2...,CC(C)N1CCN(CC1)c2ccc(cc2)/C(=C(/CCCO)\c3ccccc3...,C30H36N2O2,456.277678,0.429883,30.0,4.0,1.0,X-RAY DIFFRACTION,2.90,27.0,1.0,seed
4,Q07869,PPARA,Peroxisome proliferator-activated receptor alpha,6KB0,A,A,ITY,"icosa-5,8,11,14-tetraynoic acid",CCCCCC#CCC#CCC#CCC#CCCCC(=O)O,C20H24O2,296.177630,0.593853,20.0,2.0,1.0,X-RAY DIFFRACTION,1.35,27.0,1.0,seed


In [43]:
# Prepare input files for a single complex
df_input_subset = df_seed[df_seed["pdb_id"] == "9CUO"]

df_input_subset.to_csv(os.path.join(data_dir, f"prepare_input.csv"), index=False)

cmd = [
    "python",
    prepare_input_py,
    "--input_csv", os.path.join(data_dir, f"prepare_input.csv"),
    "--outdir", input_dir,
    "--out_csv", os.path.join(data_dir, f"prepare_input_out.csv"),
    "--evoef2_bin", EVOEF2_BIN
]
print(cmd)
# subprocess.run(cmd)


['python', '/scratch/ymeng/Neosurf_Neosurf/scripts/python/prepare_input.py', '--input_csv', '/scratch/ymeng/Neosurf_Neosurf/data/prepare_input.csv', '--outdir', '/scratch/ymeng/Neosurf_Neosurf/data/input', '--out_csv', '/scratch/ymeng/Neosurf_Neosurf/data/prepare_input_out.csv', '--evoef2_bin', '/scratch/ymeng/Neosurf_Neosurf/EvoEF2/EvoEF2']


Preparing structures:   0%|          | 0/1 [00:00<?, ?it/s]/scratch/ymeng/Neosurf_Neosurf/scripts/python/prepare_input.py:680: UserWarning: Deposited ligand resname 'A1A0J' exceeds PDB 3-character limit; using 'A1A' in HETATM records.
  placement = _get_ligand_placement(structure, pdb_ligand_chain, ligand_code)
[13:55:23] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.


Wrote /scratch/ymeng/Neosurf_Neosurf/data/prepare_input_out.csv  (1/1 rows succeeded)


[13:55:24] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
Preparing structures: 100%|██████████| 1/1 [00:05<00:00,  5.82s/it]


CompletedProcess(args=['python', '/scratch/ymeng/Neosurf_Neosurf/scripts/python/prepare_input.py', '--input_csv', '/scratch/ymeng/Neosurf_Neosurf/data/prepare_input.csv', '--outdir', '/scratch/ymeng/Neosurf_Neosurf/data/input', '--out_csv', '/scratch/ymeng/Neosurf_Neosurf/data/prepare_input_out.csv', '--evoef2_bin', '/scratch/ymeng/Neosurf_Neosurf/EvoEF2/EvoEF2'], returncode=0)

In [48]:
# Prepare input files
input_subset_dir = os.path.join(prep_input_proc_dir, "input_subsets")
os.makedirs(input_subset_dir, exist_ok=True)
output_subset_dir = os.path.join(prep_input_proc_dir, "output_subsets")
os.makedirs(output_subset_dir, exist_ok=True)

df_seed.to_csv(os.path.join(prep_input_proc_dir, "df_seed_input.csv"), index=False)

# Split df_seed into N_ARRAY_JOBS chunks
df_seed_subsets = np.array_split(df_seed, N_ARRAY_JOBS)

# Write each subset to a separate file
for i, df_seed_subset in enumerate(df_seed_subsets):
    df_seed_subset.to_csv(os.path.join(input_subset_dir, f"input_{i+1}.csv"), index=False)

# Submit slurm array job: task k reads input_k.csv, writes output_k.csv
cmd = [
    "sbatch",
    f"--array=1-{N_ARRAY_JOBS}",
    "scripts/slurm/prepare_input_array.sh",
    input_subset_dir,
    input_dir,
    output_subset_dir,
    EVOEF2_BIN,
]
print(cmd)
# subprocess.run(cmd, check=True)

/home/ymeng/miniconda3/envs/MaSIF/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:54: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


['sbatch', '--array=1-500', 'scripts/slurm/prepare_input_array.sh', '/scratch/ymeng/Neosurf_Neosurf/data/processing/1_prep_input/input_subsets', '/scratch/ymeng/Neosurf_Neosurf/data/input', '/scratch/ymeng/Neosurf_Neosurf/data/processing/1_prep_input/output_subsets', '/scratch/ymeng/Neosurf_Neosurf/EvoEF2/EvoEF2']
Submitted batch job 63492853


sbatch: [ESTIMATION] The estimated cost of this job is CHF 11.00
sbatch: ╭──────────────────────────────┬─────────────┬─────────────┬─────────────╮
sbatch: │ [in CHF]                     │ Capping     │ Consumed    │ Queued ¹⁾ ²⁾│
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ username : ymeng             │ 0           │ 144.35      │ 11.0        │
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ account : upthomae           │ 10,00

CompletedProcess(args=['sbatch', '--array=1-500', 'scripts/slurm/prepare_input_array.sh', '/scratch/ymeng/Neosurf_Neosurf/data/processing/1_prep_input/input_subsets', '/scratch/ymeng/Neosurf_Neosurf/data/input', '/scratch/ymeng/Neosurf_Neosurf/data/processing/1_prep_input/output_subsets', '/scratch/ymeng/Neosurf_Neosurf/EvoEF2/EvoEF2'], returncode=0)

0      │ 159.65      │ 11.0        │
sbatch: ╰──────────────────────────────┴─────────────┴─────────────┴─────────────╯
sbatch: ¹⁾ Estimated cost of the queued jobs and this job
sbatch: ²⁾ Queued jobs costs are based on its walltime (option --time)


In [51]:
# Gather all output .csv files into a single df_input_prepared
df_input_prepared = pd.DataFrame()
for i in range(N_ARRAY_JOBS):
    csv_path = os.path.join(output_subset_dir, f"output_{i+1}.csv")
    if os.path.exists(csv_path):
        df_input_prepared = pd.concat([df_input_prepared, pd.read_csv(csv_path)])
    else:
        print(f"Warning: {csv_path} does not exist")

print(f"df_input_prepared.shape: {df_input_prepared.shape}")

# Split into success and failed rows
df_preprocess_manifest = df_input_prepared[
    df_input_prepared['pdb_path'].notna() & df_input_prepared['ligand_path'].notna()
]
df_input_failed = df_input_prepared[
    df_input_prepared['pdb_path'].isna() | df_input_prepared['ligand_path'].isna()
]
print(f"Successfully prepared {df_preprocess_manifest.shape[0]} complexes.")
print(f"Failed to prepare input files for {df_input_failed.shape[0]} complexes.")
print(f"Failed entries:")
df_input_failed.head()


df_input_prepared.shape: (15233, 27)
Successfully prepared 14988 complexes.
Failed to prepare input files for 245 complexes.
Failed entries:


,uniprot_id,gene_name,recommendedName,pdb_id,protein_chain,ligand_chain,ligand_code,ligand_name,smiles,formula,...,neighbor_count,percent_intracellular,split,pdb_protein_chain,pdb_ligand_chain,ligand_resname,pdb_path,target,ligand,ligand_path
11,P03372,ESR1,Estrogen receptor,8VYX,C,C,A1AHU,"4,4'-[(1S,4S,5R)-5-(3,4-dihydroquinoline-1(2H)...",c1ccc2c(c1)CCCN2S(=O)(=O)[C@@H]3C[C@H]4C(=C([C...,C27H25NO5S,...,23.0,1.0,seed,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,Q96RI1,NR1H4,Bile acid receptor,3OKH,A,A,OKH,2-(4-chlorophenyl)-1-[(1S)-1-cyclohexyl-2-(cyc...,c1cc(ccc1c2nc3ccc(cc3n2[C@@H](C4CCCCC4)C(=O)NC...,C28H32ClN3O3,...,22.0,1.0,seed,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,P51449,RORC,Nuclear receptor ROR-gamma,8FB2,B,B,XO5,"(1R,15S)-16-(cyclopropylacetyl)-5-fluoro-20-me...",Cc1c2cc(cc1NS(=O)(=O)CCCCC[C@H]3C[N@](C2)CCN3C...,C22H32FN3O3S,...,21.0,1.0,seed,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,P04150,NR3C1,Glucocorticoid receptor,7PRV,B,B,GW6,"(6alpha,11alpha,14beta,16alpha,17alpha)-6,9-di...",C[C@@H]1C[C@H]2[C@@H]3C[C@@H](C4=CC(=O)C=C[C@@...,C27H29F3O6S,...,21.0,1.0,seed,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,P51449,RORC,Nuclear receptor ROR-gamma,6O3Z,A,A,LKY,3-cyano-N-(3-{[(3S)-4-(cyclopentanecarbonyl)-3...,Cc1c(cc(cc1NC(=O)c2cccc(c2)C#N)F)CN3CCN([C@H](...,C27H31FN4O2,...,20.0,1.0,seed,NaN,NaN,NaN,NaN,NaN,NaN,NaN


___
### Step 2 - Run MaSIF-preprocessing on successfully downloaded .pdb files

In [52]:
# MaSIF preprocess: split manifest into subsets and submit array job
preprocess_input_subset_dir = os.path.join(masif_preprocess_proc_dir, "input_subsets")
os.makedirs(preprocess_input_subset_dir, exist_ok=True)
preprocess_output_subset_dir = os.path.join(masif_preprocess_proc_dir, "output_subsets")
os.makedirs(preprocess_output_subset_dir, exist_ok=True)

df_preprocess_manifest.to_csv(os.path.join(masif_preprocess_proc_dir, "df_preprocess_manifest.csv"), index=False)

# Split by selecting every N_ARRAY_JOBS-th row for each subset, distributing rows in round-robin fashion
for i in range(N_ARRAY_JOBS):
    df_subset = df_preprocess_manifest.iloc[i::N_ARRAY_JOBS]
    df_subset.to_csv(
        os.path.join(preprocess_input_subset_dir, f"input_{i+1}.csv"),
        index=False,
    )

cmd = [
    "sbatch",
    f"--array=1-{N_ARRAY_JOBS}",
    "scripts/slurm/preprocess_array.sh",
    preprocess_input_subset_dir,
    preprocess_output_subset_dir,
]
print(" ".join(str(x) for x in cmd))
subprocess.run(cmd, check=True)

sbatch --array=1-500 scripts/slurm/preprocess_array.sh /scratch/ymeng/Neosurf_Neosurf/data/processing/2_masif_preprocess/input_subsets /scratch/ymeng/Neosurf_Neosurf/data/processing/2_masif_preprocess/output_subsets
Submitted batch job 63508419


sbatch: [ESTIMATION] The estimated cost of this job is CHF 132.00
sbatch: ╭──────────────────────────────┬─────────────┬─────────────┬─────────────╮
sbatch: │ [in CHF]                     │ Capping     │ Consumed    │ Queued ¹⁾ ²⁾│
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ username : ymeng             │ 0           │ 144.9       │ 132.0       │
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ account : upthomae           │ 10,000      │ 160.15      │ 132.0       │
sbatch: ╰──────────────────────────────┴─────────────┴─────────────┴─────────────╯
sbatch: ¹⁾ Estimated cost of the queued jobs and this job
sbatch: ²⁾ Queued jobs costs are based on its walltime (option --time)


CompletedProcess(args=['sbatch', '--array=1-500', 'scripts/slurm/preprocess_array.sh', '/scratch/ymeng/Neosurf_Neosurf/data/processing/2_masif_preprocess/input_subsets', '/scratch/ymeng/Neosurf_Neosurf/data/processing/2_masif_preprocess/output_subsets'], returncode=0)

In [55]:
# Gather preprocess output subsets
df_preprocess_results = pd.DataFrame()
for i in range(N_ARRAY_JOBS):
    csv_path = os.path.join(preprocess_output_subset_dir, f"output_{i+1}.csv")
    if os.path.exists(csv_path):
        df_preprocess_results = pd.concat([df_preprocess_results, pd.read_csv(csv_path)])
    else:
        print(f"Warning: {csv_path} does not exist")

print(f"df_preprocess_results.shape: {df_preprocess_results.shape}")

df_preprocess_ok = df_preprocess_results[
    df_preprocess_results["status"].isin(["success", "skipped"])
]
df_preprocess_failed = df_preprocess_results[df_preprocess_results["status"] == "error"]

print(f"Preprocessed successfully or skipped: {df_preprocess_ok.shape[0]}")
print(f"Preprocess errors: {df_preprocess_failed.shape[0]}")
print("Failed entries:")
df_preprocess_failed.head()

df_preprocess_results.shape: (14988, 29)
Preprocessed successfully or skipped: 13407
Preprocess errors: 1581
Failed entries:


,uniprot_id,gene_name,recommendedName,pdb_id,protein_chain,ligand_chain,ligand_code,ligand_name,smiles,formula,...,split,pdb_protein_chain,pdb_ligand_chain,ligand_resname,pdb_path,target,ligand,ligand_path,status,error_message
15,O60885,BRD4,Bromodomain-containing protein 4,5VBO,A,A,4K4,2-[(2-methoxy-4-{[4-(4-methylpiperazin-1-yl)pi...,CN1CCN(CC1)C2CCN(CC2)C(=O)c3ccc(c(c3)OC)Nc4ncc...,C31H38N8O3,...,seed,A,A,4K4,/scratch/ymeng/Neosurf_Neosurf/data/input/5VBO...,5VBO-4K4_A,4K4_A,/scratch/ymeng/Neosurf_Neosurf/data/input/5VBO...,error,exit code 1
19,P14324,FDPS,Farnesyl pyrophosphate synthase,4P0V,A,A,ZOL,ZOLEDRONIC ACID,c1cn(cn1)CC(O)(P(=O)(O)O)P(=O)(O)O,C5H10N2O7P2,...,seed,A,A,ZOL,/scratch/ymeng/Neosurf_Neosurf/data/input/4P0V...,4P0V-ZOL_A,ZOL_A,/scratch/ymeng/Neosurf_Neosurf/data/input/4P0V...,error,exit code 1
8,P24941,CDK2,Cyclin-dependent kinase 2,5ANG,A,A,WY3,7-HYDROXY-4-(MORPHOLINOMETHYL)CHROMEN-2-ONE,c1cc2c(cc1O)OC(=O)C=C2CN3CCOCC3,C14H15NO4,...,seed,A,A,WY3,/scratch/ymeng/Neosurf_Neosurf/data/input/5ANG...,5ANG-WY3_A,WY3_A,/scratch/ymeng/Neosurf_Neosurf/data/input/5ANG...,error,exit code 1
11,P24941,CDK2,Cyclin-dependent kinase 2,3S0O,A,A,50Z,"[4-amino-2-(prop-2-en-1-ylamino)-1,3-thiazol-5...",C=CCNc1nc(c(s1)C(=O)c2ccccn2)N,C12H12N4OS,...,seed,A,A,50Z,/scratch/ymeng/Neosurf_Neosurf/data/input/3S0O...,3S0O-50Z_A,50Z_A,/scratch/ymeng/Neosurf_Neosurf/data/input/3S0O...,error,exit code 1
22,P16152,CBR1,Carbonyl reductase [NADPH] 1,2PFG,A,A,DDD,"(5R,10S)-5-{[(CARBOXYMETHYL)AMINO]CARBONYL}-7-...",C1CC(=O)N2C[N@]([C@@H]1C(=O)O)CSC[C@H]2C(=O)NC...,C12H17N3O6S,...,seed,A,A,DDD,/scratch/ymeng/Neosurf_Neosurf/data/input/2PFG...,2PFG-DDD_A,DDD_A,/scratch/ymeng/Neosurf_Neosurf/data/input/2PFG...,error,exit code 1


___
### Step 3 - Run masif_search around ligand residues

In [56]:
# seed_id is the canonical pipeline identifier computed by prepare_input.
# Use the manifest target column directly instead of re-deriving from seed fields.
assert df_preprocess_ok["target"].notna().all(), "Some successful rows are missing target"
df_preprocess_ok["seed_id"] = df_preprocess_ok["target"]
df_preprocessed_seeds = df_preprocess_ok[df_preprocess_ok["split"] == "seed"]            # seed-ligand complexes
df_preprocessed_targets = df_preprocess_ok[df_preprocess_ok["split"] == "target"]        # known E3 ligase-ligand complexes
seed_ids = df_preprocessed_seeds["seed_id"].tolist()
print("Number of seed ids: ", len(seed_ids))
seed_ids[0:5]

Number of seed ids:  13140


/tmp/ipykernel_687309/766793879.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_preprocess_ok["seed_id"] = df_preprocess_ok["target"]


['7QPN-DVF_B', '4M12-1YZ_A', '2IW6-QQ2_C', '8V4X-A1AAB_A', '5XXJ-8HF_A']

#### Submit array job to search using all available unique target complexes

In [57]:
# Deduplicate target complexes to unique ligase-compound complexes, keeping the row with the highest resolution (lowest numeric value)
print(f"df_seed_targets.shape: {df_preprocessed_targets.shape}")
df_preprocessed_targets_sorted = df_preprocessed_targets.sort_values(by="resolution", ascending=True)
df_preprocessed_targets_dedup = df_preprocessed_targets_sorted.drop_duplicates(subset=["uniprot_id", "ligand_code"], keep="first")
print(f"df_seed_targets_dedup.shape: {df_preprocessed_targets_dedup.shape}")
df_preprocessed_targets_dedup.head()

df_seed_targets.shape: (267, 30)
df_seed_targets_dedup.shape: (128, 30)


,uniprot_id,gene_name,recommendedName,pdb_id,protein_chain,ligand_chain,ligand_code,ligand_name,smiles,formula,...,pdb_protein_chain,pdb_ligand_chain,ligand_resname,pdb_path,target,ligand,ligand_path,status,error_message,seed_id
20,P41182,BCL6,B-cell lymphoma 6 protein,7OKL,A,A,VJ5,2-chloranyl-4-[[1-methyl-2-oxidanylidene-4-[[(...,C[C@H](c1ncccn1)NC2=CC(=O)N(c3c2cc(cc3)Nc4ccnc...,C22H18ClN7O,...,A,A,VJ5,/scratch/ymeng/Neosurf_Neosurf/data/input/7OKL...,7OKL-VJ5_A,VJ5_A,/scratch/ymeng/Neosurf_Neosurf/data/input/7OKL...,success,NaN,7OKL-VJ5_A
27,P41182,BCL6,B-cell lymphoma 6 protein,7LWF,A,A,YNA,N-(3-chloropyridin-4-yl)-2-[5-(3-cyano-4-hydro...,CN1C=Nc2c(c(cn2CC(=O)Nc3ccncc3Cl)c4ccc(c(c4)C#...,C21H15ClN6O3,...,A,A,YNA,/scratch/ymeng/Neosurf_Neosurf/data/input/7LWF...,7LWF-YNA_A,YNA_A,/scratch/ymeng/Neosurf_Neosurf/data/input/7LWF...,success,NaN,7LWF-YNA_A
8,P62942,FKBP1A,Peptidyl-prolyl cis-trans isomerase FKBP1A,9LYG,A,A,A1L7S,5-[(2~{S})-1-cyclohexylsulfonylpiperidin-2-yl]...,COc1ccc(cc1OC)CCCc2nc(on2)[C@@H]3CCCCN3S(=O)(=...,C24H35N3O5S,...,A,A,A1L,/scratch/ymeng/Neosurf_Neosurf/data/input/9LYG...,9LYG-A1L7S_A,A1L_A,/scratch/ymeng/Neosurf_Neosurf/data/input/9LYG...,success,NaN,9LYG-A1L7S_A
21,P41182,BCL6,B-cell lymphoma 6 protein,7RV3,A,A,7R5,N-(3-chloropyridin-4-yl)-2-[2-(morpholin-4-yl)...,c1cncc(c1NC(=O)Cn2ccc3c2N=C(NC3=O)N4CCOCC4)Cl,C17H17ClN6O3,...,A,A,7R5,/scratch/ymeng/Neosurf_Neosurf/data/input/7RV3...,7RV3-7R5_A,7R5_A,/scratch/ymeng/Neosurf_Neosurf/data/input/7RV3...,success,NaN,7RV3-7R5_A
26,P41182,BCL6,B-cell lymphoma 6 protein,7ZWX,A,A,KA0,"6-[1,3-benzodioxol-5-ylmethyl(methyl)amino]-1-...",CC(C)(C)n1c2c(cn1)C(=O)NC(=N2)N(C)Cc3ccc4c(c3)...,C18H21N5O3,...,A,A,KA0,/scratch/ymeng/Neosurf_Neosurf/data/input/7ZWX...,7ZWX-KA0_A,KA0_A,/scratch/ymeng/Neosurf_Neosurf/data/input/7ZWX...,success,NaN,7ZWX-KA0_A


In [ ]:
import os
import numpy as np
import subprocess

# Function to submit a search job (array of targets on array of seeds)
def submit_neosurf_search(query_targets, seed_ids, n_array_jobs, masif_search_out_dir, dry_run=True):
    """
    Writes query_targets.txt and splits seed_ids for SLURM array search job.
    Submits search_array.sh as an array job.

    Args:
        query_targets (list): List of query target IDs.
        seed_ids (list): List of seed IDs to split for subsets.
        n_array_jobs (int): Number of array jobs/chunks to split seed_ids into.
        masif_search_out_dir (str): Output directory for search results and intermediate files.
    """

    master_subset_dir = os.path.join(masif_search_out_dir, 'subset')
    os.makedirs(master_subset_dir, exist_ok=True)
    query_target_txt = os.path.join(masif_search_out_dir, 'query_targets.txt')
    
    # Ensure query_targets is a list
    if not isinstance(query_targets, (list, np.ndarray)):
        query_targets = [query_targets]

    if not dry_run:
        with open(query_target_txt, 'w') as f:
            for target in query_targets:
                f.write(f"{target}\n")

    if not dry_run:
        seed_chunks = np.array_split(seed_ids, n_array_jobs)
        for idx, chunk in enumerate(seed_chunks):
            subset_file = os.path.join(master_subset_dir, f"{idx+1}")
            with open(subset_file, "w") as sf:
                for seed in chunk:
                    sf.write(f"{seed}\n")

    cmd = [
        "sbatch",
        f"--array=1-{n_array_jobs}",
        "scripts/slurm/search_array.sh",
        query_target_txt,
        masif_search_out_dir,
        master_subset_dir,
    ]
    print(" ".join(str(x) for x in cmd))

    if not dry_run:
        subprocess.run(cmd, check=True)

# Example usage for a single target
query_targets = df_preprocessed_targets[df_preprocessed_targets["pdb_id"] == "8VLB"].iloc[0]["seed_id"]

print(f"Number of query targets: {len(query_targets)}")
print(f"Number of seed ids: {len(seed_ids)}")
print(f"query_targets: {query_targets}")

# submit_neosurf_search(query_targets, seed_ids, N_ARRAY_JOBS, masif_search_out_dir, dry_run=False)

Number of query targets: 10
Number of seed ids: 13140
query_targets: 8VLB-3JF_A
sbatch --array=1-500 scripts/slurm/search_array.sh /scratch/ymeng/Neosurf_Neosurf/data/masif_search/query_targets.txt /scratch/ymeng/Neosurf_Neosurf/data/masif_search /scratch/ymeng/Neosurf_Neosurf/data/masif_search/subset
Submitted batch job 63738667


sbatch: [ESTIMATION] The estimated cost of this job is CHF 66.00
sbatch: ╭──────────────────────────────┬─────────────┬─────────────┬─────────────╮
sbatch: │ [in CHF]                     │ Capping     │ Consumed    │ Queued ¹⁾ ²⁾│
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ username : ymeng             │ 0           │ 160.5       │ 66.0        │
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ account : upthomae           │ 10,000      │ 176.3       │ 66.0        │
sbatch: ╰──────────────────────────────┴─────────────┴─────────────┴─────────────╯
sbatch: ¹⁾ Estimated cost of the queued jobs and this job
sbatch: ²⁾ Queued jobs costs are based on its walltime (option --time)


In [67]:
# Submit search job for all targets
query_targets = df_preprocessed_targets_dedup["seed_id"].tolist()
submit_neosurf_search(query_targets, seed_ids, N_ARRAY_JOBS, masif_search_out_dir, dry_run=False)

sbatch --array=1-500 scripts/slurm/search_array.sh /scratch/ymeng/Neosurf_Neosurf/data/masif_search/query_targets.txt /scratch/ymeng/Neosurf_Neosurf/data/masif_search /scratch/ymeng/Neosurf_Neosurf/data/masif_search/subset
Submitted batch job 63739974


sbatch: [ESTIMATION] The estimated cost of this job is CHF 66.00
sbatch: ╭──────────────────────────────┬─────────────┬─────────────┬─────────────╮
sbatch: │ [in CHF]                     │ Capping     │ Consumed    │ Queued ¹⁾ ²⁾│
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ username : ymeng             │ 0           │ 160.65      │ 66.0        │
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ account : upthomae           │ 10,000      │ 176.45      │ 66.0        │
sbatch: ╰──────────────────────────────┴─────────────┴─────────────┴─────────────╯
sbatch: ¹⁾ Estimated cost of the queued jobs and this job
sbatch: ²⁾ Queued jobs costs are based on its walltime (option --time)


____

In [4]:
# Build a lookup from target id -> manifest fields so we never need to parse
# pdb_id or chain IDs out of the target string (which is not safe for entries
# with multi-letter author chains or ligand codes containing underscores).
_target_lookup = df_preprocess_ok.set_index("target")[
    ["pdb_id", "pdb_protein_chain", "pdb_ligand_chain"]
].to_dict("index")

def _get_target_info(target_id):
    """Return (pdb_id, chain_list) for a target id using the manifest lookup.

    Falls back to splitting on the last underscore for targets not in the
    manifest (e.g. manually specified query targets).
    """
    if target_id in _target_lookup:
        info = _target_lookup[target_id]
        chains = [info["pdb_protein_chain"]]
        if info["pdb_ligand_chain"] != info["pdb_protein_chain"]:
            chains.append(info["pdb_ligand_chain"])
        return info["pdb_id"], chains
    # Fallback: parse {identifier}_{chains} — last segment is chain suffix
    parts = target_id.rsplit("_", 1)
    chains = list(parts[1]) if len(parts) > 1 else []
    return parts[0], chains

def print_pymol_commands(row):
    matched_protein = row["matched_protein"]
    matched_patch_id = row["matched_patch_id"]
    target_protein = row["target"]
    flattened_transform = row["flattened_transform"]

    target_pdb, target_chain_list = _get_target_info(target_protein)
    matched_pdb, matched_chain_list = _get_target_info(matched_protein)

    target_chains_str = "chain " + " chain ".join(target_chain_list)
    matched_chains_str = "chain " + " chain ".join(matched_chain_list)

    # Load target protein
    print(f"fetch {target_pdb}, {target_protein}_{matched_protein}_{matched_patch_id}")
    print(f"remove {target_protein}_{matched_protein}_{matched_patch_id} AND (not {target_chains_str})")

    # load and transform matched protein
    print(f"fetch {matched_pdb}, {matched_protein}_{matched_patch_id}")
    print(f"remove {matched_protein}_{matched_patch_id} AND (not {matched_chains_str})")
    print(f"apply_transform {matched_protein}_{matched_patch_id}, '{flattened_transform}'")

    # Copy transformed matched_protein to target protein object
    print(f"copy_to {target_protein}_{matched_protein}_{matched_patch_id}, {matched_protein}_{matched_patch_id}")

    # Delete {matched_protein}_{matched_patch_id} object
    print(f"delete {matched_protein}_{matched_patch_id}")

# Apply to all rows
df_dedup.apply(print_pymol_commands, axis=1)

fetch 6H0F, 6H0F_B_7AFW_A_80
remove 6H0F_B_7AFW_A_80 AND (not chain B)
fetch 7AFW, 7AFW_A_80
remove 7AFW_A_80 AND (not chain A)
apply_transform 7AFW_A_80, '-0.7859344656941732,-0.5043491230556201,-0.35768558498636877,-3.6363662083943424,0.11463107809434332,-0.6873130377377955,0.717259021616719,-111.81487360429189,-0.6075909245281254,0.5227167016928679,0.597997088790896,33.085055390718644,0.0,0.0,0.0,1.0'
copy_to 6H0F_B_7AFW_A_80, 7AFW_A_80
delete 7AFW_A_80
fetch 6H0G, 6H0G_B_7AFW_A_51
remove 6H0G_B_7AFW_A_51 AND (not chain B)
fetch 7AFW, 7AFW_A_51
remove 7AFW_A_51 AND (not chain A)
apply_transform 7AFW_A_51, '0.8182982172028582,-0.553642071949644,-0.1544942843277068,-72.9274821077419,0.46224671737335316,0.4741008655351706,0.7493706303134404,35.656988642990825,-0.341637234504941,-0.6846231265930978,0.6438751234002663,-102.12152938615822,0.0,0.0,0.0,1.0'
copy_to 6H0G_B_7AFW_A_51, 7AFW_A_51
delete 7AFW_A_51
fetch 7LPS, 7LPS_B_7AFW_A_70
remove 7LPS_B_7AFW_A_70 AND (not chain B)
fetch 7AFW,

9     None
15    None
28    None
43    None
59    None
66    None
71    None
75    None
dtype: object

In [43]:
def print_pymol_commands(row):
    matched_protein = row["matched_protein"]
    matched_patch_id = row["matched_patch_id"]
    flattened_transform = row["flattened_transform"]

    # Use the manifest lookup defined in the cell above for safe pdb_id extraction.
    matched_pdb, matched_chain_list = _get_target_info(matched_protein)
    matched_chains_str = "chain " + " chain ".join(matched_chain_list)

    print(f"fetch {matched_pdb}, {matched_protein}_{matched_patch_id}")
    #print(f"select {matched_protein}_{matched_patch_id} AND (not {matched_chains_str})")
    #print('cmd.remove("sele");cmd.delete("sele")')
    print(f"apply_transform {matched_protein}_{matched_patch_id}, '{flattened_transform}'")

# Apply to all rows
df_dedup.apply(print_pymol_commands, axis=1)

fetch 5QSQ, 5QSQ_B_40
apply_transform 5QSQ_B_40, '0.18811655710121694,0.3201757470768955,0.9284932158762048,-63.34899149991804,0.1194347315502052,-0.945812608023888,0.30195008760154796,-21.857826309349356,0.9748576849181194,0.05409252708834745,-0.21616311588539106,-29.57452413756281,0.0,0.0,0.0,1.0'
fetch 5QSV, 5QSV_D_29
apply_transform 5QSV_D_29, '-0.8848374941118574,-0.459948308188821,-0.07423047088689808,-154.06565366628317,0.4264906636583265,-0.8637764209055722,0.2683207194754801,4.137717891846782,-0.18753219143957434,0.20576163024675115,0.96046542295497,101.9424492820446,0.0,0.0,0.0,1.0'
fetch 6M92, 6M92_CA_58
apply_transform 6M92_CA_58, '0.34474053707697766,0.7252827786602123,-0.5959184953286809,-67.98240992873171,-0.5430922038973354,-0.3636905840683252,-0.7568223154254735,-60.38063982437132,-0.7656401375070488,0.5845460204628572,0.2685165354298039,-40.661634546822384,0.0,0.0,0.0,1.0'
fetch 6M92, 6M92_CA_183
apply_transform 6M92_CA_183, '-0.054155842613027305,-0.8787217692945033,

0    None
1    None
2    None
3    None
4    None
5    None
6    None
7    None
8    None
dtype: object